# Pet edges: v3 + v5 ensemble + ViTMatte

The best segmentation so far is test-time augmentation (TTA) on v3 and v5: v5 + TTA is best overall, v3 + TTA is clearly
better on small pets. This notebook **averages their pet probabilities** and uses that as the mask for soft edges:

1. **v3 + TTA** (256 px) and **v5 + TTA** (384 px): each runs at flip × 3 scales (0.75, 1, 1.25 of its training size);
   the pet probabilities are resized to the original image and averaged, first over TTA, then over the two models
   (`ENSEMBLE_WEIGHTS`).
2. An **automatic trimap** around the 0.5-thresholded ensemble mask: shrunk = *sure pet*, grown = *sure background*,
   the ring in between = *unknown*.
3. **ViTMatte** (`hustvl/vitmatte-small-composition-1k`) turns the unknown ring into a soft **alpha matte**.

**Checks.**
* Section 4b: v3 + TTA vs v5 + TTA vs the ensemble on **every** test image at full resolution (cleaned trimaps,
  boundary excluded), overall, by pet size and for the hardest breeds.
* Section 5: on `EVAL_N` test images, the ensemble + ViTMatte must not lower IoU, and its soft pixels should land inside
  the annotated boundary band (compared with a blurred-mask baseline).
* Visuals, cut-outs and `cutout(path)` for any photo; everything is saved to Drive.

**Runtime:** T4 GPU, *Run all*. About 15–20 min (the full-test segmenter comparison is most of it).

## 1. Setup

In [ ]:
import os, sys, random, shutil, tarfile, urllib.request, zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from scipy import ndimage
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from tqdm.auto import tqdm
from transformers import VitMatteImageProcessor, VitMatteForImageMatting

IN_COLAB = "google.colab" in sys.modules
print("torch", torch.__version__)

EVAL_N = 600                 # test images used for the ViTMatte edge metrics in 5 (None = all)
BAND_FRAC = 0.02             # unknown-ring half-width as a fraction of the image's long side
BAND_MIN_PX = 4              # ... but at least this many pixels
MATTE_MODEL = "hustvl/vitmatte-small-composition-1k"
SHOW_BREEDS = ["Persian", "samoyed", "keeshond", "Maine_Coon", "leonberger", "Sphynx"]
SEED = 42
REQUIRE_GPU = True

DATA_DIR = Path("/content/dataset")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/upgaming/Animal Classification & Segmentation")
SEG_MODELS = {"v3": DRIVE_PROJECT_DIR / "checkpoints_v3" / "best.pt",
              "v5": DRIVE_PROJECT_DIR / "checkpoints_v5" / "best.pt"}
ENSEMBLE_WEIGHTS = {"v3": 0.5, "v5": 0.5}
TTA_SCALES = [0.75, 1.0, 1.25]
SEG_EVAL_N = None            # images for the segmenter comparison in 4b (None = the whole test set)
OUT_DIR = Path("/content/edges")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError("No GPU found. In Colab choose Runtime ▸ Change runtime type ▸ T4 GPU, then run all again.")
else:
    device = torch.device("cpu")
    print("Running on CPU")
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## 2. Data

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

IMAGES_DIR = DATA_DIR / "images"
ANN_DIR = DATA_DIR / "annotations"
TRIMAPS_DIR = ANN_DIR / "trimaps"
ARCHIVES = {
    "images.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/images.tar.gz",
    "annotations.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/annotations.tar.gz",
}


def dataset_ready():
    return IMAGES_DIR.is_dir() and TRIMAPS_DIR.is_dir() and (ANN_DIR / "trainval.txt").exists()


def extract(archive):
    print("extracting", archive.name)
    with tarfile.open(archive) as t:
        try:
            t.extractall(DATA_DIR, filter="data")
        except TypeError:  # Python without tarfile filters
            t.extractall(DATA_DIR)


if not dataset_ready():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    drive_data = DRIVE_PROJECT_DIR / "Data"
    for name, url in ARCHIVES.items():
        local = DATA_DIR / name
        on_drive = drive_data / name
        if on_drive.exists():
            print("copying", on_drive)
            shutil.copy(on_drive, local)
        else:
            print("downloading", url)
            urllib.request.urlretrieve(url, local)
        extract(local)
        local.unlink()
assert dataset_ready(), f"dataset not found in {DATA_DIR}"

PET, BG, BND = 1, 2, 3


def read_list(path):
    return [l.split()[0] for l in path.read_text().splitlines() if l.strip() and not l.startswith("#")]


def breed_of(name):
    return name.rsplit("_", 1)[0]


CLEAN_TRIMAPS = ANN_DIR / "trimaps_clean"
audit_zips = sorted((DRIVE_PROJECT_DIR / "runs").glob("audit_*/trimaps_clean.zip"))
if not (CLEAN_TRIMAPS.is_dir() and any(CLEAN_TRIMAPS.glob("*.png"))) and audit_zips:
    with zipfile.ZipFile(audit_zips[-1]) as z:
        z.extractall(ANN_DIR)
LABEL_DIR = CLEAN_TRIMAPS if CLEAN_TRIMAPS.is_dir() and any(CLEAN_TRIMAPS.glob("*.png")) else TRIMAPS_DIR
print("labels from:", LABEL_DIR)


def load_trimap(name):
    return np.array(Image.open(LABEL_DIR / f"{name}.png"))


def load_rgb(name):
    return np.array(Image.open(IMAGES_DIR / f"{name}.jpg").convert("RGB"))


test_names = [n for n in read_list(ANN_DIR / "test.txt")
              if not n.startswith("._") and (IMAGES_DIR / f"{n}.jpg").exists() and (TRIMAPS_DIR / f"{n}.png").exists()
              and (np.array(Image.open(TRIMAPS_DIR / f"{n}.png")) == PET).any()]
eval_names = test_names if EVAL_N is None else sorted(random.Random(SEED).sample(test_names, min(EVAL_N, len(test_names))))
seg_eval_names = test_names if SEG_EVAL_N is None else sorted(random.Random(SEED).sample(test_names, min(SEG_EVAL_N, len(test_names))))
print(f"test images: {len(test_names)} | segmenter comparison: {len(seg_eval_names)} | ViTMatte metrics: {len(eval_names)}")

## 3. Load v3, v5 and ViTMatte

In [ ]:
def conv_bn_relu(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class DecoderBlock(nn.Module):
    def __init__(self, c_in, c_skip, c_out):
        super().__init__()
        self.block = nn.Sequential(conv_bn_relu(c_in + c_skip, c_out), conv_bn_relu(c_out, c_out))

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.block(x)


class UNetResNet34(nn.Module):
    def __init__(self, num_classes, pretrained=False):
        super().__init__()
        r = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.dec0 = DecoderBlock(32, 0, 16)
        self.head = nn.Conv2d(16, num_classes, 1)

    def forward(self, x):
        s0 = self.stem(x)
        e1 = self.enc1(self.pool(s0))
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d = self.dec4(e4, e3)
        d = self.dec3(d, e2)
        d = self.dec2(d, e1)
        d = self.dec1(d, s0)
        return self.head(self.dec0(d))


segs = {}
for name, path in SEG_MODELS.items():
    ck = torch.load(path, map_location=device, weights_only=False)
    assert ck["config"]["boundary_mode"] == "ignore" and len(ck["class_names"]) == 2, f"{name}: expected a boundary-ignored model"
    net = UNetResNet34(2).to(device).eval()
    net.load_state_dict(ck["model"])
    segs[name] = (net, ck["config"]["img_size"])
    print(f"{name} loaded: epoch {ck['epoch']}, input {ck['config']['img_size']}px")
assert abs(sum(ENSEMBLE_WEIGHTS.values()) - 1) < 1e-6

processor = VitMatteImageProcessor.from_pretrained(MATTE_MODEL)
matte = VitMatteForImageMatting.from_pretrained(MATTE_MODEL).to(device).eval()
print("ViTMatte loaded:", MATTE_MODEL)

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

## 4. The pipeline: v3 + v5 (TTA) → trimap → alpha
`model_probs` runs each model with TTA at its own training size and resizes the pet probability to the original
resolution; `pet_probability` averages them with `ENSEMBLE_WEIGHTS`.
`auto_trimap` turns the 0.5-thresholded mask into sure-pet / unknown / sure-background with a ring whose half-width
scales with the image (`BAND_FRAC`, min `BAND_MIN_PX`). `alpha_matte` runs ViTMatte on the image + trimap.

In [ ]:
@torch.no_grad()
def tta_prob(net, native, x, h, w):
    acc, n = 0, 0
    for s in TTA_SCALES:
        sz = max(64, int(round(native * s / 32)) * 32)
        xi = (F.interpolate(x, size=(sz, sz), mode="bilinear", antialias=True, align_corners=False) - MEAN) / STD
        for flip in (False, True):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                lo = net(xi.flip(-1) if flip else xi).float()
            lo = lo.flip(-1) if flip else lo
            acc = acc + F.interpolate(lo, size=(h, w), mode="bilinear", align_corners=False).softmax(1)[0, 1]
            n += 1
    return (acc / n).cpu().numpy()


def model_probs(img):
    h, w = img.shape[:2]
    x = torch.from_numpy(img).permute(2, 0, 1)[None].float().div(255).to(device)
    return {name: tta_prob(net, native, x, h, w) for name, (net, native) in segs.items()}


def pet_probability(img):
    return sum(p * ENSEMBLE_WEIGHTS[m] for m, p in model_probs(img).items())


def band_px(shape):
    return max(BAND_MIN_PX, int(round(BAND_FRAC * max(shape))))


def auto_trimap(mask):
    r = band_px(mask.shape)
    disk = np.hypot(*np.mgrid[-r:r + 1, -r:r + 1]) <= r
    sure_fg = ndimage.binary_erosion(mask, disk)
    sure_bg = ~ndimage.binary_dilation(mask, disk)
    tri = np.full(mask.shape, 128, np.uint8)
    tri[sure_fg], tri[sure_bg] = 255, 0
    return tri


@torch.no_grad()
def alpha_matte(img, tri):
    inputs = processor(images=Image.fromarray(img), trimaps=Image.fromarray(tri), return_tensors="pt").to(device)
    a = matte(**inputs).alphas[0, 0, :img.shape[0], :img.shape[1]].float().cpu().numpy()
    a[tri == 255], a[tri == 0] = 1.0, 0.0      # keep the sure regions exactly as the segmenter decided
    return a.clip(0, 1)


def feathered(mask):
    """Baseline soft edge: the ensemble mask blurred with a Gaussian of the same width as the trimap ring."""
    return ndimage.gaussian_filter(mask.astype(np.float32), sigma=band_px(mask.shape) / 2)


def run(name):
    img = load_rgb(name)
    p = pet_probability(img)
    mask = p > 0.5
    tri = auto_trimap(mask)
    return img, p, mask, tri, alpha_matte(img, tri)

## 4b. Which segmenter? v3 + TTA vs v5 + TTA vs the ensemble (full test, full resolution)
Every official test image (empty masks dropped), at its **original resolution**, against the cleaned trimaps, boundary excluded.
Pet IoU is reported over the whole test set (all pixels pooled) and per image, plus by pet size and for the hardest breeds.

In [ ]:
SIZE_BINS = [0, 0.10, 0.30, 0.50, 1.0]
SIZE_LABELS = ["<10%", "10–30%", "30–50%", ">50%"]
variants = list(SEG_MODELS) + ["ensemble"]
pooled = {v: [0, 0] for v in variants}   # intersection, union
seg_rows = []
for n in tqdm(seg_eval_names, desc="segmenters on the full test"):
    img, t = load_rgb(n), load_trimap(n)
    probs = model_probs(img)
    probs["ensemble"] = sum(probs[m] * w for m, w in ENSEMBLE_WEIGHTS.items())
    valid, gt = t != BND, t == PET
    r = {"filename": n, "breed": breed_of(n), "pet_share": (t != BG).mean()}
    for v in variants:
        p = (probs[v] > 0.5) & valid
        g = gt & valid
        i, u = (p & g).sum(), (p | g).sum()
        pooled[v][0] += i
        pooled[v][1] += u
        r[v] = i / u if u else 1.0
    seg_rows.append(r)
seg_res = pd.DataFrame(seg_rows)
seg_res["size_bin"] = pd.cut(seg_res.pet_share, SIZE_BINS, labels=SIZE_LABELS, include_lowest=True)

seg_table = pd.DataFrame({v: {"pet IoU (whole test set)": pooled[v][0] / pooled[v][1],
                              "pet IoU (per-image mean)": seg_res[v].mean(),
                              **{f"pet IoU {l}": seg_res.loc[seg_res.size_bin == l, v].mean() for l in SIZE_LABELS},
                              "images": len(seg_res)} for v in variants})
seg_table.columns = [f"{c} + TTA" if c in SEG_MODELS else "v3 + v5 ensemble" for c in seg_table.columns]
display(seg_table.round(4))
hard = seg_res.groupby("breed")[variants].mean().sort_values(variants[0]).head(6)
print("hardest breeds (by the first model):")
display(hard.round(4))
best_seg = seg_table.loc["pet IoU (whole test set)"].idxmax()
print(f"best segmenter: {best_seg}")

## 5. Does it help? Metrics on test images
Per image, against the annotated trimap:

* `iou_mask`, `iou_matte`: pet IoU on **non-boundary** pixels for the ensemble mask and for `alpha > 0.5`.
* `soft_in_band`: of the soft pixels (0.05 < alpha < 0.95), the share inside the annotated boundary band, for ViTMatte and
  for the feathered baseline. Higher = the soft edge is where the annotators said the edge is.
* `band_covered`: share of the annotated band that the automatic trimap marks unknown (does the ring cover the real edge?).
* `band_soft`: share of annotated-band pixels that get a partial alpha instead of a hard 0/1.

In [ ]:
def iou(pred, t):
    valid = t != BND
    p, g = pred & valid, (t == PET) & valid
    u = (p | g).sum()
    return (p & g).sum() / u if u else 1.0


def soft(a):
    return (a > 0.05) & (a < 0.95)


rows = []
for n in tqdm(eval_names, desc="ensemble + ViTMatte"):
    img, p, mask, tri, a = run(n)
    t = load_trimap(n)
    fa = feathered(mask)
    band = t == BND
    s_m, s_f = soft(a), soft(fa)
    rows.append({"filename": n, "breed": breed_of(n), "species": "cat" if n[0].isupper() else "dog",
                 "iou_mask": iou(mask, t), "iou_matte": iou(a > 0.5, t), "iou_feathered": iou(fa > 0.5, t),
                 "soft_in_band_matte": band[s_m].mean() if s_m.any() else np.nan,
                 "soft_in_band_feathered": band[s_f].mean() if s_f.any() else np.nan,
                 "band_covered": (tri[band] == 128).mean() if band.any() else np.nan,
                 "band_soft_matte": s_m[band].mean() if band.any() else np.nan,
                 "band_soft_feathered": s_f[band].mean() if band.any() else np.nan})
res = pd.DataFrame(rows)

summary = pd.DataFrame({
    "ensemble mask": [res.iou_mask.mean(), np.nan, np.nan],
    "ensemble + feathered": [res.iou_feathered.mean(), res.soft_in_band_feathered.mean(), res.band_soft_feathered.mean()],
    "ensemble + ViTMatte": [res.iou_matte.mean(), res.soft_in_band_matte.mean(), res.band_soft_matte.mean()],
}, index=["pet IoU, non-boundary pixels (mean)", "soft pixels inside annotated band", "annotated band given soft alpha"])
display(summary.round(4))
print(f"automatic trimap ring covers {res.band_covered.mean():.1%} of the annotated band on average "
      f"(ring half-width {BAND_FRAC:.0%} of the long side, min {BAND_MIN_PX}px)")
print(f"IoU change from matting: {res.iou_matte.mean() - res.iou_mask.mean():+.4f} "
      f"(worse in {(res.iou_matte < res.iou_mask - 0.01).mean():.1%} of images by >0.01)")
display(res.groupby("species")[["iou_mask", "iou_matte", "soft_in_band_matte", "soft_in_band_feathered"]].mean().round(4))

## 6. What the edges look like
Image | v3 mask | automatic trimap | ViTMatte alpha | cut-out on a flat colour. The last row zooms into the edge.

In [ ]:
def checker(h, w, s=16):
    c = ((np.arange(h)[:, None] // s + np.arange(w)[None] // s) % 2).astype(np.float32)
    return (0.75 + 0.2 * c)[..., None].repeat(3, -1)


def composite(img, a, bg_rgb=(40, 110, 200)):
    bg = np.ones_like(img, np.float32) * np.array(bg_rgb, np.float32)
    return (img * a[..., None] + bg * (1 - a[..., None])).astype(np.uint8)


picks = []
for b in SHOW_BREEDS:
    cand = [n for n in eval_names if breed_of(n) == b] or [n for n in test_names if breed_of(n) == b]
    if cand:
        picks.append(cand[0])
picks = picks or eval_names[:4]

fig, axes = plt.subplots(len(picks), 5, figsize=(22, 4.4 * len(picks)), squeeze=False)
for row, n in zip(axes, picks):
    img, p, mask, tri, a = run(n)
    row[0].imshow(img); row[0].set_title(n)
    row[1].imshow(mask, cmap="gray"); row[1].set_title("ensemble mask (hard)")
    row[2].imshow(tri, cmap="gray", vmin=0, vmax=255); row[2].set_title("automatic trimap")
    row[3].imshow(a, cmap="gray", vmin=0, vmax=1); row[3].set_title("ViTMatte alpha")
    row[4].imshow(composite(img, a)); row[4].set_title("cut-out")
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def edge_crop(mask, size):
    """A square crop centred on a point of the mask outline with the most edge around it."""
    edge = mask & ~ndimage.binary_erosion(mask)
    dens = ndimage.uniform_filter(edge.astype(np.float32), size)
    y, x = np.unravel_index(np.argmax(dens), dens.shape)
    h, w = mask.shape
    y0, x0 = np.clip(y - size // 2, 0, max(h - size, 0)), np.clip(x - size // 2, 0, max(w - size, 0))
    return slice(y0, y0 + size), slice(x0, x0 + size)


fig, axes = plt.subplots(len(picks), 4, figsize=(16, 4.2 * len(picks)), squeeze=False)
for row, n in zip(axes, picks):
    img, p, mask, tri, a = run(n)
    ys, xs = edge_crop(mask, max(64, min(img.shape[:2]) // 3))
    hard = composite(img, mask.astype(np.float32))
    feath = composite(img, feathered(mask))
    soft_ = composite(img, a)
    row[0].imshow(img[ys, xs]); row[0].set_title(f"{n}: edge zoom")
    row[1].imshow(hard[ys, xs]); row[1].set_title("ensemble hard mask")
    row[2].imshow(feath[ys, xs]); row[2].set_title("ensemble feathered")
    row[3].imshow(soft_[ys, xs]); row[3].set_title("ensemble + ViTMatte")
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Use it on any photo
`cutout(path)` returns the alpha matte and an RGBA PNG (transparent background) for any image file.

In [ ]:
def cutout(path, save_to=None):
    img = np.array(Image.open(path).convert("RGB"))
    mask = pet_probability(img) > 0.5
    a = alpha_matte(img, auto_trimap(mask))
    rgba = np.dstack([img, (a * 255).astype(np.uint8)])
    if save_to:
        Image.fromarray(rgba).save(save_to)
    return a, rgba


example = IMAGES_DIR / f"{picks[0]}.jpg"
a, rgba = cutout(example, OUT_DIR / f"{picks[0]}_cutout.png")
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(checker(*a.shape)); ax[0].imshow(rgba); ax[0].set_title("RGBA cut-out (checkerboard = transparent)")
ax[1].imshow(a, cmap="gray"); ax[1].set_title("alpha")
for x in ax:
    x.axis("off")
plt.show()

## 8. Save to Drive

In [ ]:
res.to_csv(OUT_DIR / "edge_metrics_per_image.csv", index=False)
summary.to_csv(OUT_DIR / "edge_metrics_summary.csv")
seg_table.to_csv(OUT_DIR / "segmenter_comparison_full_test.csv")
seg_res.to_csv(OUT_DIR / "segmenter_per_image_full_test.csv", index=False)
alpha_dir = OUT_DIR / "alpha"
alpha_dir.mkdir(exist_ok=True)
for n in tqdm(picks + eval_names[:200], desc="saving alpha mattes"):
    img, p, mask, tri, a = run(n)
    Image.fromarray((a * 255).astype(np.uint8)).save(alpha_dir / f"{n}.png")
    if n in picks:
        Image.fromarray(np.dstack([img, (a * 255).astype(np.uint8)])).save(OUT_DIR / f"{n}_cutout.png")
with zipfile.ZipFile(OUT_DIR / "alpha_mattes.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p_ in sorted(alpha_dir.glob("*.png")):
        z.write(p_, f"alpha/{p_.name}")
shutil.rmtree(alpha_dir)

if IN_COLAB and DRIVE_PROJECT_DIR.exists():
    run_dir = DRIVE_PROJECT_DIR / "runs" / datetime.now().strftime("edges_ensemble_%Y%m%d_%H%M%S")
    shutil.copytree(OUT_DIR, run_dir)
    print("saved to", run_dir)
else:
    print("results kept in", OUT_DIR)